## Header

In [ ]:
%load_ext tensorboard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import networkx as nx
from icecream import ic
from gymnasium import Env, spaces
import time
from io import BytesIO
import PIL
from IPython.display import display, Image

## Other class


In [ ]:
class Car(object):
    def __init__(self,start, target):
        self.pos = tuple(start)
        self.target = tuple(target)
        self.path = []
        self.distance = 0
    def get_path(self,G):
        if self.target is None:
            raise Exception("No target to get_path")
        # maze = obs*1
        # maze[self.pos[0]][self.pos[1]] = 2
        # maze[self.target[0]][self.target[1]] = 2
        self.path = search(G, self.pos, self.target)
        # ic(obs,maze,self.path)
        return self.path
    def move(self):
        self.distance+=1
        self.pos = self.path[1]
    def get_pos(self):
        return self.pos
class Home(object):
    def __init__(self,x,y):
        self.pos = (x,y)
        self.target = None
        
    def __repr__(self):
        return f"Home {self.pos}->{self.target}"
    def get_pos(self):
        return self.pos
    def set_target(self, t):
        if not isinstance(t,Target):
            ic(t)
            raise TypeError("The giving t is not target class")
        self.target = t.pos
class Target(object):
    def __init__(self,x,y):
        self.pos = (x,y)
    
    def __repr__(self):
        return f"Target {self.pos}"

def search(G, source, target):
    source = tuple(source)
    target = tuple(target)
    if not nx.has_path(G,source, target):
        return None
    else:
        path = nx.astar_path(G,source, target)
        return path
# subax1 = plt.subplot()
# nx.spectral_layout(G)
# nx.draw_spectral(G)   # default spring_layout

def ask_direction(pos, ref):
    dif = [pos[0]-ref[0], pos[1]-ref[1]]
    if dif == [-1,0]:
        return 2 #up
    elif dif == [0,-1]:
        return  1 #left
    elif dif == [0,1]:
        return -2 #right
    elif dif == [1,0]:
        return -1 #down
    #opposite direction -> product<0
    else:
        raise ValueError('Error: direction is incorrect')


In [ ]:
def display_img(im):
    bio = BytesIO()
    im.save(bio, format='png')
    display(Image(bio.getvalue(), format='png'))

# Environment

## Main Env

In [ ]:
class townScape(Env):
    def __init__(self, shape=(10,10,)):
        super(townScape, self).__init__()

        if not isinstance(shape, tuple) or len(shape) != 2:
            raise ValueError("Environment: Incorrect shape")
        shape = tuple(shape)
        self.observation_shape = shape
        self.observation_space = spaces.Box(low=0, high = 4, shape = tuple(shape), dtype = np.uint8)
        # self.observation_space = spaces.MultiDiscrete(np.ones(shape, dtype= np.uint8) * 5)
        self.obs = self.observation_space.sample() * 0
        #agent can not destroy building

        max_act_code = np.array([self.observation_shape[0]-1, self.observation_shape[1]-1],dtype= np.uint8)
        self.action_space = spaces.Discrete(self.act_encoder(max_act_code))
        
        self.roadG = nx.create_empty_copy(nx.grid_2d_graph(*shape, periodic=False, create_using=nx.DiGraph))
        
    def get_obs_meaning(self):
        return {"0":"ground","1":"obstacle","2":"road","3":"home","4":"target"}
    def get_act_meaning(self):
        return {"0":"replace to road","1":"replace to home","2":"replace to target"}
    def get_ht_info(self):
        return (len(self.home_list), len(self.target_list))
    def act_encoder(self,act):
        nActType = 2
        shape = self.observation_shape
        if len(act) != 2:
            raise ValueError("Action should be list of length 2")
        for i in range(2):
            # if isinstance(act[i],np.uint8):
            #     raise ValueError("Action index{} is not int".format(i))
            if(act[i]>=shape[i]):
                raise ValueError("Action index{} out of range".format(i))
            
        message = act[0]*shape[1]+act[1]
        return message
    def act_decoder(self, code):
        nActType = 2
        shape = self.observation_shape
        max_code = self.act_encoder([shape[0]-1, shape[1]-1])
        if (code<0) or (code> max_code):
            raise ValueError("Code Out of Range")
        act = [code//shape[1], code%shape[1]]
        return tuple(act)
    

## Reset

In [ ]:
def reset(self, requirements = {'map':None, 'home': 0, 'target': 0}, seed=None, options=None):
    super(townScape, self).reset(seed=seed)
    #version3: only obstacle can be initial state
    self.obs = self.observation_space.sample() * 0
    self.total_road = 0
    self.total_time = 0
    self.total_used_road = 0
    self.phase = 1 #1->build home/target #2->build road

    self.home_list = []
    self.target_list = []
    self.road_list = []
    self.car_list = []
    self.car_space = []
    
    shape = self.observation_shape
    self.roadG = nx.create_empty_copy(nx.grid_2d_graph(*shape , periodic=False, create_using=nx.DiGraph))
    self.used_road = [[0 for j in range(self.observation_shape[1])] for i in range(self.observation_shape[0])]
    self.has_ent = np.zeros(shape, dtype = bool)

    r,c = self.observation_shape
    area = r*c

    #home and target init
    n_home = int(requirements['home'])
    n_target = int(requirements['target'])
    
    desired_h = 0.13 #probability of requirement h
    desired_t = 0.07 #probability of requirement h
    scale = area//100
    
    if n_home == 0:
        n_home = int(np.random.normal(desired_h*area, scale))
        n_home = max(1,n_home)
    if n_target == 0:
        n_target = int(np.random.normal(desired_t*area, scale))
        n_target = max(1,n_target)

    self.rem_home = n_home
    self.rem_target = n_target
    custom_map = requirements['map'] 
    if isinstance(custom_map, np.ndarray):
        self.obs = np.array(custom_map).astype(np.uint8)
    elif custom_map == None:
        # full random
        desired_obstacle = 0.03
        n_obstacle = int(np.random.normal(desired_obstacle*area, area//100))
        n_obstacle = max(1,n_obstacle)

        pos_vector = [(i,j) for i in range(r) for j in range(c)]
        pos_index = np.arange(len(pos_vector))

        pos_obstacle = np.random.choice(pos_index, n_obstacle, replace = False)
        for ele in pos_obstacle: 
            self.obs[pos_vector[ele]] = 1
            # pos_index = pos_index[pos_index!=ele]
        self.obs = self.obs.astype(np.uint8) 
    elif custom_map == []:
        n_obstacle = 0
    else:
        raise ValueError("custom_map is either numpy.ndarray, [], or None")

    # pairing
    # ran_t_list = np.random.choice(self.target_list,len(self.target_list),replace = False)
    # for i,h in enumerate(self.home_list):
    #     if i<len(self.target_list):
    #         h.set_target(ran_t_list[i])
    #     else:
    #         h.set_target(np.random.choice(self.target_list))

    # path_info = self.bfs()
    # temp = []
    # for i in path_info:
    #     if i[1] != -1:
    #         temp.append(i[0])
    # self.home_list = temp

    return self.obs, {}

townScape.reset = reset

## Step

In [ ]:
def step(self,actionCode,debug= False):
    # ic(self.total_time)
    self.total_time+=1
    area = self.observation_shape[0] * self.observation_shape[1]
    action = self.act_decoder(actionCode)
    row = action[0]
    col = action[1]
    pos = row,col
    reward = 0.0
    suc_r = 0

    info = {}
    is_bad_action = False
    if self.obs[row][col] != 0:
        reward = area*(-30.0)
        is_bad_action = True
        info['action'] = 'bad'
    
    else:    
        if self.rem_home == 0 and self.rem_target == 0:
            if self.phase == 1:
                self.phase = 2
                # pairing
                ran_t_list = np.random.choice(self.target_list,len(self.target_list),replace = False)
                for i,h in enumerate(self.home_list):
                    if i<len(self.target_list):
                        h.set_target(ran_t_list[i])
                    else:
                        h.set_target(np.random.choice(self.target_list))

                path_info = self.bfs()
                for h,val in path_info:
                    if val == -1:
                        self.home_list.remove(h)
                if debug:
                    ic(self.home_list)

            info['action'] = 'road'
            self.obs[row][col] = 2
            self.total_road+=1
            self.road_list.append((row,col))
            self.addEdgetoG(row,col)
            reward += self.zone_road((row,col))
        elif self.rem_home >  0: 
            info['action'] = 'home'
            self.obs[row,col] = 3
            self.rem_home -= 1
            self.home_list.append(Home(row,col))
            self.addEdgetoG(row,col)
            reward += self.zone_home((row,col))
        elif self.rem_target > 0:
            info['action'] = 'target'
            self.obs[row,col] = 4
            self.rem_target -= 1
            self.target_list.append(Target(row,col))
            self.addEdgetoG(row,col)
            reward += self.zone_target((row,col))

    if self.phase == 2 and not is_bad_action:
        #Reward calculation 
        tf, suc_c, suc_r, over_road = self.simulate()
        web, iso_h_ratio, iso_t_ratio = self.connectivity()
        if(web==0): 
            web = 10

        if tf == "blank" or len(tf) == 0:
            sim_reward = 0
        else:
            sim_reward = np.median(np.array(tf))*300+ suc_c*50
        
        con_reward = max(0,6-web)*20 
        con_reward += iso_h_ratio*30+iso_t_ratio*30
        
        # path_info = self.bfs()
        # path_reward = 0
        # for i in path_info:
        #     if i[1]==-1:
        #         if debug:
        #             ic(i[0])
        #         continue
        #     path_reward -= i[1]
        
        #path_reward/=n_home
        
        reward += sim_reward + con_reward

        #debug
        if debug:
            ic(reward, sim_reward, con_reward, self.total_time)

    # n_home, n_target = self.get_ht_info()
    truncated = False
    if self.total_time>=1.0*area:
        done = True
        info['status'] = 'timed out'
        if debug: 
            ic('timed out')
    elif suc_r>=0.9 :
        reward+=area*10.0
        done = True
        info['status'] = 'success'
        if debug:
            ic('success')
    elif is_bad_action:
        done = False
        info['status'] = 'bad action'
        if debug:
            ic('Bad action',reward,self.total_time)
    elif self.phase == 1:
        done = False
        info['status'] = 'phase 1'
        if debug:
            ic(info['action'], pos, reward,self.total_time)
    else:
        done = False
        info['status'] = 'phase 2'
    #observation reward terminate truncuate info
    '''terminated= True if environment terminates (eg. due to task completion, failure etc.)
    truncated= True if episode truncates due to a time limit or a reason that is not defined as part of the task MDP.'''
    # if done:
    #     ic('done',self.total_time, 1.2*area)
    return self.obs, reward, done, truncated, info

townScape.step = step

In [ ]:
def addEdgetoG(self,r,c):
        # ic("adding edge", r, c ,act)
        for m in [[-1,0],[0,-1],[0,1],[1,0]]: 
            i,j = r+m[0], c+m[1]
            # ic(r,c,self.obs[r,c])
            if (0<=i and i< self.obs.shape[0]) and (0<=j and j< self.obs.shape[1]):
                if(self.obs[r,c]==2) and self.obs[i,j]==2:
                    self.roadG.add_edge((r,c),(i,j))
                    self.roadG.add_edge((i,j),(r,c))
                elif(self.obs[r,c]==2) and self.obs[i,j]==3:
                    self.roadG.add_edge((i,j),(r,c))
                elif(self.obs[r,c]==3) and self.obs[i,j]==2:
                    self.roadG.add_edge((r,c),(i,j))
                elif(self.obs[r,c]==2) and self.obs[i,j]==4:
                    self.roadG.add_edge((r,c),(i,j))
                elif(self.obs[r,c]==4) and self.obs[i,j]==2:
                    self.roadG.add_edge((i,j),(r,c))
townScape.addEdgetoG = addEdgetoG

def zone_road(self,pos):
    M = [(-1,0),(0,-1),(0,1),(1,0)]
    point = 0
    for m in M:
        cur = pos[0] + m[0], pos[1] + m[1]
        i,j = cur
        if (0<=i and i< self.obs.shape[0]) and (0<=j and j< self.obs.shape[1]):
            if not self.has_ent[cur]:
                self.has_ent[cur] = True
                point += 10
    return point
townScape.zone_road = zone_road

def zone_home(self,pos):
    M55 = [(-2, -2), (-2, -1),(-2, 0),(-2, 1),(-2, 2),(-1, -2),(-1, 2),(0, -2),(0, 2),(1, -2),(1, 2),(2, -2),(2, -1),(2, 0),(2, 1),(2, 2)]
    for m in M55:
        cur = pos[0] + m[0], pos[1] + m[1]
        i,j = cur
        if (0<=i and i< self.obs.shape[0]) and (0<=j and j< self.obs.shape[1]):
            if self.obs[cur] == 3:
                return 20
    return 0
townScape.zone_home = zone_home

def zone_target(self,pos):
    M33 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    M55 = [(-2, -2), (-2, -1),(-2, 0),(-2, 1),(-2, 2),(-1, -2),(-1, 2),(0, -2),(0, 2),(1, -2),(1, 2),(2, -2),(2, -1),(2, 0),(2, 1),(2, 2)]
    M77 = [(-3, -3),(-3, -2),(-3, -1),(-3, 0),(-3, 1),(-3, 2),(-3, 3),(-2, -3),(-2, 3),(-1, -3),(-1, 3),(0, -3),(0, 3),(1, -3),(1, 3),(2, -3),(2, 3),(3, -3),(3, -2),(3, -1),(3, 0),(3, 1),(3, 2),(3, 3)]
    point = 0
    for m in M55:
        cur = pos[0] + m[0], pos[1] + m[1]
        i,j = cur
        if (0<=i and i< self.obs.shape[0]) and (0<=j and j< self.obs.shape[1]):
            if self.obs[cur] == 4:
                point += 20
    
    for m in M33:
        cur = pos[0] + m[0], pos[1] + m[1]
        i,j = cur
        if (0<=i and i< self.obs.shape[0]) and (0<=j and j< self.obs.shape[1]):
            if self.obs[cur] == 3:
                return point-30
    for m in M55:
        cur = pos[0] + m[0], pos[1] + m[1]
        i,j = cur
        if (0<=i and i< self.obs.shape[0]) and (0<=j and j< self.obs.shape[1]):
            if self.obs[cur] == 3:
                return point-20
    for m in M77:
        cur = pos[0] + m[0], pos[1] + m[1]
        i,j = cur
        if (0<=i and i< self.obs.shape[0]) and (0<=j and j< self.obs.shape[1]):
            if self.obs[cur] == 3:
                return point-10
    
    return point
townScape.zone_target = zone_target


In [ ]:
# M33 = [(i,j) for i in range(-1,2) for j in range(-1,2) ]
# M33.remove((0,0))
# M55 = [(i,j) for i in range(-2,3) for j in range(-2,3) ]
# M77 = [(i,j) for i in range(-3,4) for j in range(-3,4) ]
# for m in M55:
#     M77.remove(m)
# for m in M33:
#     M55.remove(m)
# M55.remove((0,0))

# for m in M77:
#     print(m, end = ',')

## Render

In [ ]:
def render(self,mode = 'RGB'):
    v = list(self.observation_shape)
    u = v+[3]
    u = np.array(u)
    u = u*32
    u[2]/=32    
    img = PIL.Image.new('RGB', (int(u[1]), int(u[0])))
    pic0 = PIL.Image.open('../src/assets/Grass.png')
    pic1 = PIL.Image.open('../src/assets/Obstacle.png')
    pic2 = PIL.Image.open('../src/assets/Road.png')
    pic3 = PIL.Image.open('../src/assets/Home.png')
    pic4 = PIL.Image.open('../src/assets/Building.png')
    pic5 = PIL.Image.open('../src/assets/Car.png')
    pic6 = PIL.Image.open('../src/assets/CarKee.png')
    for i in range(v[0]):
        for j in range(v[1]):
            if self.obs[i][j] == 0:
                img.paste(pic0, (32*j,32*i))
            elif self.obs[i][j] == 1:
                img.paste(pic1, (32*j,32*i))
            elif self.obs[i][j] == 2:
                img.paste(pic2, (32*j,32*i))
            elif self.obs[i][j] == 3:
                img.paste(pic3, (32*j,32*i))
            elif self.obs[i][j] == 4:
                img.paste(pic4, (32*j,32*i))
    for c in self.car_list:
        pos = c.get_pos()
        i,j = pos
        if self.obs[i][j] == 2:
            if len(self.car_space[i][j]) == 1:
                img.paste(pic5, (32*j,32*i))
            elif len(self.car_space[i][j]) == 2:
                img.paste(pic6, (32*j,32*i))
    #for PILLOW
    return img

    #for cv2 in google colab
    open_cv_image = np.array(img) 
    # Convert RGB to BGR 
    open_cv_image = open_cv_image[:, :, ::-1].copy() 
    return open_cv_image
townScape.render = render

## Connectivity



In [ ]:
from collections import deque
def connectivity(self):
    maze = self.obs * 1
    h_list = [h.pos for h in self.home_list]
    t_list = [t.pos for t in self.target_list]
    r_list = [r for r in self.road_list]
    n_graph = 0
    visited = []
    while len(r_list)>0:
        n_graph += 1
        q = deque()
        q.append(r_list[0])
        r_list.pop(0)
        while len(q)>0:
            cur_r, cur_c = q.popleft()
            visited.append((cur_r, cur_c))
            move = [(-1,0),(0,-1),(0,1),(1,0)]
            # ic(cur)
            for i in move:
                r,c = cur_r+i[0], cur_c+i[1]
                # ic(r,c)
                if (0<=r and r<self.observation_shape[0]) and (0<=c and c<self.observation_shape[1]) and (r,c) not in visited:
                    if maze[r,c] == 2:
                        # ic('Road')
                        maze[r,c] = 0
                        q.append((r,c))
                        if (r,c) in r_list: r_list.remove((r,c))
                    elif maze[r,c] == 3:
                        # ic('home')
                        if (r,c) in h_list: h_list.remove((r,c))
                        visited.append((r,c))
                    elif maze[r,c] == 4:
                        # ic('target')
                        if (r,c) in t_list: t_list.remove((r,c))
                        visited.append((r,c))
    # return {'n_graph':n_graph, 'iso_h_ratio': len(h_list)/len(self.home_list), 'iso_t_ratio':len(t_list)/len(self.target_list)}
    if len(self.home_list) == 0:
        iso_h_rate = 0
    else:
        iso_h_rate = len(h_list)/len(self.home_list)
    if len(self.target_list) == 0:
        iso_t_rate = 0
    else:
        iso_t_rate = len(t_list)/len(self.target_list)
    return n_graph, iso_h_rate, iso_t_rate

townScape.connectivity = connectivity

## target bfs

In [ ]:
from queue import PriorityQueue
def bfs(self):
    maze = self.obs * 1
    
    result = []
    M = [(-1,0), (0,-1),(0,1),(1,0)]
    for h in self.home_list:
        q = PriorityQueue()
        s = h.pos
        t = h.target
        q.put((0,s))
        visited = [s]
        f = 0
        while not q.empty() and f==0:
            g,cur =  q.get()
            for i,j in M:
                r, c = cur[0]+i,  cur[1]+j
                if 0<=r and r<self.observation_shape[0] and (0<=c and c<self.observation_shape[1]) and (r,c) not in visited:
                    if (r,c) == t:
                        result.append((h, g))
                        f = 1
                        break
                    else:
                        if maze[r,c]==2:
                            q.put((g, (r,c)))
                            visited.append((r,c))
                        elif maze[r,c]==0:
                            q.put((g+1, (r,c)))
                            visited.append((r,c))
        if f==0:
            result.append((h, -1))
    return result
townScape.bfs = bfs

## Simulate


In [ ]:
def simulate(self,render=False,debug = False):
    
    time = 0
    self.car_list = []
    traffic_flow = []
    success_cnt = 0 #number of car that can be travel to
    # nopath_factor = -0.2
    # blankArea_factor = -0.5
    
    if len(self.home_list)*len(self.target_list) == 0:
        return "blank", 0, 0, 0
    elif self.phase == 1:
        return "phase1", 0, 0, 0

    self.car_space = [[list() for j in range(self.observation_shape[1])] for i in range(self.observation_shape[0])]
    for i,h in enumerate(self.home_list):

        # t = random.choice(self.target_list)

        # c = Car(tuple(h.pos), tuple(h.target))
        c = Car(h.pos, h.target)

        if c.get_path(self.roadG) != None:
            self.car_list.append(c)
            self.car_space[h.pos[0]][h.pos[1]].append(c)
        else:
            continue
    f = 0
    while(len(self.car_list)>0):
        time+=1
        # ic(time)
        tf = 0.0
        temp = []
        moved = 0
        for i,c in enumerate(self.car_list):
            c.get_path(self.roadG)
            # willremove = False
            cur = c.path[0]
            # ic(c.pos, c.path)
            if(cur == c.target): #finish
                self.car_space[cur[0]][cur[1]].remove(c)
                success_cnt+=1
            else:
                # ic(c.path)
                next = c.path[1]
                if len(self.car_space[next[0]][next[1]]) == 0:
                    c.move()
                    if(self.used_road[next[0]][next[1]]==0 and self.obs[next[0],next[1]] == 2):
                        self.used_road[next[0]][next[1]]= 1 
                        self.total_used_road+=1
                        # ic(next[0],next[1])
                    moved+=1
                    self.car_space[cur[0]][cur[1]].remove(c)
                    self.car_space[next[0]][next[1]].append(c)
                    # c.get_path(self.obs)
                elif len(self.car_space[next[0]][next[1]])==1 and self.obs[next[0]][next[1]] != 4:
                    other_c = self.car_space[next[0]][next[1]][0]
                    # ic(c.pos, other_c.pos)
                    # img = self.render()
                    # cv2_imshow(img)
                    # ic(self.obs)
                    # ic(c.path, other_c.path)
                    
                    if ask_direction(other_c.path[1],other_c.path[0])*ask_direction(c.path[1],c.path[0])==-2: #ERROR: index out of range
                        c.move()
                        if(self.used_road[next[0]][next[1]]==0 and self.obs[next[0],next[1]] == 2):
                            self.used_road[next[0]][next[1]]= 1 
                            self.total_used_road+=1
                            # ic(next[0],next[1])
                        moved+=1
                        self.car_space[cur[0]][cur[1]].remove(c)
                        self.car_space[next[0]][next[1]].append(c)
                        # c.get_path(self.obs)
            # if not willremove:
                temp.append(c)
        # if moved == 0:
        #     raise RuntimeError('Error: No movement')
        tf += moved/len(self.car_list)
        if not self.total_road :
            tf*= len(self.car_list) / 1
        else:
            tf *= len(self.car_list) / self.total_road
        
        self.car_list = temp
        traffic_flow.append(tf)
        if render:
            img = self.render()
            display_img(img)
            ic("---------------")
        # ic(tf)
        if(moved == 0) and len(self.car_list)>1: 
            if f == 1:
                for c in self.car_list:
                    ic(c.pos, c.path)
                display_img(self.render())
                ic(self.roadG.edges)
                raise RuntimeError("Timed out no movement")
            else:
                f = 1
        else: f = 0
    # return {"tf": traffic_flow, "success": success_cnt, "success_rate": success_cnt/min(len(self.home_list), len(self.target_list))}
    # ic([traffic_flow, success_cnt, success_cnt/min(len(self.home_list),len(self.target_list)), self.total_road-self.total_used_road])
    over_road = self.total_road-self.total_used_road
    return [traffic_flow, success_cnt, success_cnt/len(self.home_list), over_road]
    # return [pre_rew, traffic_flow] #reward and traffic flow list
townScape.simulate = simulate

## Utitlity

In [ ]:
def set_target(self, t_list):
    if len(t_list)!=len(self.home_list):
        raise ValueError("unmatched t_list to home_list")
    for i,h in enumerate(self.home_list):
        h.set_target(Target(*t_list[i]))
townScape.set_target = set_target

# Testing

In [ ]:
shape = (5,5)
env = townScape(shape)
req = {'map': None, "home":6, "target":6}
for _ in range(300):
    env.reset(req)
    done = False
    i = 0
    while not done:
        action =  env.action_space.sample()
        obs, reward, done, _, info = env.step(action, debug =False)
        # img = env.render()
    res = env.simulate()
# env.simulate(1)



In [ ]:
ic(env.home_list)
ic(env.target_list)
ic(env.phase, info)

In [ ]:
shape = (5,8)
env = townScape(shape)
map = [[0,0,0,0,0,0,0,0],
       [0,0,0,0,0,0,0,0],
       [0,0,0,0,0,0,0,0],
       [1,0,0,0,0,0,0,0],
       [0,1,0,0,0,0,0,0]]
req = {'map': np.array(map), "home":3, "target":2}
env.reset(req)
done = False
i = 0
# env.simulate()
h_list = ((1,2), (3,2),(4,0))
t_list = ((1,6), (3,6))
r_list = ((1,3),(1,4),(1,5), (2,4), (3,3),(3,4),(3,5))

act_list = h_list + t_list + r_list

for act in act_list:
    action = env.act_encoder(act) 
    obs, reward, done, _, info = env.step(action, debug =False)
    ic(reward)
ic(done)
ic(info)
ic(env.simulate())
display_img(env.render())

In [ ]:
#put here
shape = (5,6)
env = townScape(shape)

map = [[0,0,0,0,0,4],
       [4,0,0,3,0,0],
       [0,0,0,0,0,0],
       [0,3,0,0,0,0],
       [0,0,0,3,0,0]]
map = np.array(map,dtype =np.uint8)

act = [[0,0,0,0,0,0],
       [0,0,0,0,0,1],
       [0,0,0,0,0,0],
       [0,0,1,1,0,1],
       [0,0,0,0,0,0],]
act = np.array(act,dtype =np.uint8)
env.reset(requirements = {'map': map, 'home':0, 'target': 0})
for i in range(shape[0]):
    for j in range(shape[1]):
        if act[i,j]:
            action = (i, j)
            _, reward, _, _, _ = env.step(env.act_encoder(action))
            # cv2_imshow(env.render())
            # ic(reward)
            # ic("---------------")
# env.obs[obstacle_arr] = 1
# t_list = [[5,1], [4,3], [4,7], [1,7]]
display_img(env.render())
ic(env.home_list)
path_info = env.bfs()

# Check ENv

In [ ]:
from stable_baselines3.common import env_checker
env2 = townScape()
env_checker.check_env(env2)

# Train Model

In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import time


In [ ]:
T = int(time.time())
model_dir = f"../training/PPO-{T}"
log_dir = f"../training/logs/PPO-{T}"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

obs_shape = (10,10)
env = townScape(obs_shape)
# env = DummyVecEnv([lambda: env])
env.reset()
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log = log_dir)

In [ ]:
%reload_ext tensorboard

In [ ]:
TIMESTEPS = 2048 #total timesteps per ep
train_eps = 10
for ep in range(1,train_eps+1):
    model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps = False, tb_log_name = "LOG",progress_bar = False)
model.save(f"{model_dir}/{T}_{obs_shape}_{train_eps}eps")
env.close()
del model

In [ ]:
# !rm -rf ./.logs/

In [ ]:
training_log_path = f"{log_dir}/LOG_0"
%tensorboard --logdir {training_log_path}

# Test model

In [ ]:
#test model
env = townScape((10,10))
model = PPO.load(f"{model_dir}/{T}_{obs_shape}_{train_eps}eps")

eps = 3
for ep in range(1,1+eps):
    obs, _ = env.reset({"map":None, "home":0, "target":0})
    # ic(env.obs)
    done = False
    ic('load')
    while not done:
        action, _ = model.predict(obs) #it returns action, state
        obs, reward, done, _, info = env.step(action, debug = False)
    
    display_img(env.render())
    tf,_,_,_ = env.simulate()
    median_of_tf = np.median(tf)
    ic(median_of_tf)
    ic(info)
env.close()

In [ ]:
#saved model to drive
from datetime import date
model = PPO.load(f"{model_dir}/{T}_{obs_shape}_{train_eps}eps")
today = date.today()
model_name = 'M1-005'
model.save(f"../training/{today}/{model_name}/model{T}_{obs_shape}_{train_eps}eps")

#  f"/content/drive/MyDrive/Krokect/Training/{today}/log{T}_{obs_shape}_{train_eps}eps"
# print(f"/content/drive/MyDrive/Krokect/Training/{today}/model{T}_{obs_shape}_{train_eps}eps")

In [ ]:
# ic(training_log_path)
training_log_path = f"{log_dir}/LOG_0"
saved_log_path = f"../training/{today}/{model_name}/log/"
!xcopy {training_log_path} {saved_log_path} /E

In [ ]:
!touch 123.txt

In [ ]:
!mv 123.txt 456.txt

# User

In [ ]:
shape = (10,10)
env = townScape(shape)
#map can be 0,1,3,4 1=obstacle 3=home 4=target
map = [[0,0,0,0,0,1,0,0,0,0],
       [1,0,0,1,0,0,0,0,0,0],
       [0,0,0,0,0,0,0,0,0,0],
       [0,0,0,1,0,0,0,0,0,0],
       [1,1,1,0,0,1,1,1,1,1],
       [0,0,0,0,0,0,0,0,0,0],
       [0,1,0,1,0,0,0,0,0,0],
       [0,0,1,0,0,0,0,0,0,0],
       [0,1,0,1,0,0,0,0,0,0],
       [0,1,0,1,0,0,0,0,0,0]]
map = np.array(map,dtype =np.uint8)

# #desired act(road); in case you want to test reward+other function
# act = [[0,0,0,0,0,0],
#        [0,1,0,0,0,1],
#        [0,0,0,0,0,0],
#        [0,0,0,0,0,0],
#        [0,0,1,0,0,1],]
# act = np.array(act,dtype =np.uint8)

'''
map->   None: random everything
        []: no obstacle
        custom_map
home/target-> will be random to desired number in the dict requirements as below if didn't put custom_map
                if put custom_map: this home/target will be ignored
'''
this_requirements = {'map': map, 'home':1, 'target': 1}
env.reset(requirements = this_requirements) 

ic('initial state')
display_img(env.render())

#display home_list and their fixed target
# ic(env.home_list)

In [ ]:
print("Loading...")
env = townScape((10,10))
model = PPO.load(f"{model_dir}/{T}_{obs_shape}_{train_eps}eps")

eps = 10
bad_act_count = [0.0 for i in range(eps)]
eps_score = []
timed_out_r = 0
for ep in range(eps):
    obs, _ = env.reset(this_requirements)
    done = False
    while not done:
        action, _ = model.predict(obs) #it returns action, state
        obs, reward, done, _, info = env.step(action)
        if info['action'] == 'bad':
            bad_act_count[ep] +=1
        # eps_score[ep] +=reward
    # bad_act_count[ep]
    # cv2_imshow(env.render())
    if info['status'] == 'timed out':
        timed_out_r += 1
    else:
        ic(ep, 'success')
        display_img(env.render())

timed_out_r /= eps

ic('Result')
ic(timed_out_r)
ic(eps_score)
ic(bad_act_count)


# env.simulate(render = True)

In [ ]:
info = {'status': 'timed out'}
while info['status'] == 'success':
    info['status'] = 'success'